# 🤝 05 — Ensemble Pondéré & Comparaison Finale
## Weighted Blending + Diebold-Mariano Test

Ce notebook importe les prédictions de tous les modèles (CSV), crée un ensemble pondéré par l'inverse du MAPE, et compare les performances.

## 1. 📦 Imports & Chargement des prédictions

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Charger le test (vérité terrain)
test = pd.read_csv("test_data.csv", index_col=0, parse_dates=True)
y_true = test["revenue"]

# Charger les prédictions de chaque modèle
predictions = {}
model_files = {
    "SARIMA": "predictions_sarima.csv",
    "Holt-Winters": "predictions_holtwinters.csv",
    "XGBoost": "predictions_xgboost.csv",
    "Prophet": "predictions_prophet.csv",
}

for name, file in model_files.items():
    try:
        df_pred = pd.read_csv(file, parse_dates=["date"]).set_index("date")
        predictions[name] = df_pred["prediction"]
        print(f"✅ {name:<15} → chargé ({len(df_pred)} mois)")
    except FileNotFoundError:
        print(f"⚠️ {name:<15} → fichier non trouvé")

print(f"\n✅ Test (réel) : {len(y_true)} mois")

## 2. 📐 Calcul des Métriques Individuelles

In [ ]:
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true.values - y_pred.values) / y_true.values)) * 100

def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denom != 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

def mase(y_true, y_pred, y_train):
    naive = np.mean(np.abs(np.diff(y_train, 12)))
    return np.mean(np.abs(y_true - y_pred)) / naive if naive != 0 else np.nan

results = []
for name, pred in predictions.items():
    common = y_true.index.intersection(pred.index)
    yt, yp = y_true[common], pred[common]
    mae = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mape_val = mape(yt, yp)
    smape_val = smape(yt, yp)
    r2 = r2_score(yt, yp)
    results.append({"Modèle": name, "MAE (€)": mae, "RMSE (€)": rmse,
                    "MAPE (%)": mape_val, "sMAPE (%)": smape_val, "R²": r2})

df_results = pd.DataFrame(results).set_index("Modèle").round({"MAE (€)": 0, "RMSE (€)": 0, "MAPE (%)": 2, "R²": 4})
df_results = df_results.sort_values("MAPE (%)")
print("📊 Résultats individuels :")
print(df_results.to_string())

## 3. 🤝 Ensemble Pondéré (Weighted Blending)

In [ ]:
# Poids = inverse du MAPE
weights = {r["Modèle"]: 1.0 / max(r["MAPE (%)"], 0.1) for r in results}
total_w = sum(weights.values())
for k in weights:
    weights[k] /= total_w

print("⚖️ Poids de l'ensemble :")
for name, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"   {name:<15} : {w:.1%}")

# Prédiction ensemble
common_idx = y_true.index
ensemble_pred = np.zeros(len(common_idx))
for name, pred in predictions.items():
    ensemble_pred += weights[name] * pred[common_idx].values

# Métriques ensemble
mae_ens = mean_absolute_error(y_true, ensemble_pred)
rmse_ens = np.sqrt(mean_squared_error(y_true, ensemble_pred))
mape_ens = mape(y_true, pd.Series(ensemble_pred, index=common_idx))
r2_ens = r2_score(y_true, ensemble_pred)

print(f"\n📊 Ensemble Pondéré")
print(f"   MAE  : {mae_ens:>12,.0f} €")
print(f"   RMSE : {rmse_ens:>12,.0f} €")
print(f"   MAPE : {mape_ens:>11.2f} %")
print(f"   R²   : {r2_ens:>11.4f}")

# Ajouter au tableau
df_results.loc["Ensemble Pondéré"] = [mae_ens, rmse_ens, mape_ens, np.nan, r2_ens]
df_results = df_results.sort_values("MAPE (%)")

print(f"\n🥇 Meilleur modèle : {df_results.index[0]} (MAPE: {df_results.iloc[0]['MAPE (%)']:.2f}%)")

## 4. 🔬 Diebold-Mariano Test

In [ ]:
def diebold_mariano(y_true, y_pred1, y_pred2):
    d = np.abs(y_true - y_pred1) - np.abs(y_true - y_pred2)
    d = d[~np.isnan(d)]
    if len(d) < 2: return 1.0, 0.0
    mean_d, var_d = np.mean(d), np.var(d, ddof=1) / len(d)
    if var_d <= 0: return 1.0, 0.0
    dm_stat = mean_d / np.sqrt(var_d)
    return dm_stat, 2 * (1 - stats.norm.cdf(abs(dm_stat)))

print("🔬 Diebold-Mariano Test (p-values)")
print("   H0: Les deux modèles ont la même précision")
print("   p < 0.05 → différence significative\n")

best_name = df_results.index[0]
best_key = next((k for k in predictions if k in best_name), list(predictions.keys())[0])
y_best = predictions[best_key].values

for other_name in df_results.index[1:]:
    if other_name == "Ensemble Pondéré":
        y_other = ensemble_pred
    else:
        other_key = next((k for k in predictions if k in other_name), None)
        if other_key is None: continue
        y_other = predictions[other_key].values
    
    dm_stat, p_val = diebold_mariano(y_true.values, y_best, y_other)
    result = "✅ Significatif" if p_val < 0.05 else "⚠️ Non significatif"
    print(f"   {best_name[:20]:<20} vs {other_name[:20]:<20} → p={p_val:.3f} {result}")

## 5. 📊 Graphique Comparatif Final

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, figure=fig)

# 1. Toutes les prévisions
ax1 = fig.add_subplot(gs[0, :])
y_true.plot(ax=ax1, label="✅ Réel 2022", color="#1B5E20", linewidth=3, zorder=5)
colors_pred = ["#E65100", "#F57C00", "#6A1B9A", "#7B1FA2"]
for (name, pred), color in zip(predictions.items(), colors_pred):
    pred.plot(ax=ax1, label=name, color=color, linestyle="--", linewidth=2, alpha=0.85)
pd.Series(ensemble_pred, index=common_idx).plot(ax=ax1, label="Ensemble", color="#C62828", linewidth=2.5)
ax1.set_title("Prévisions 2022 — Tous les Modèles vs Réel", fontsize=14, fontweight="bold")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M€"))
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

# 2. MAPE
ax2 = fig.add_subplot(gs[1, 0])
mape_vals = df_results["MAPE (%)"]
colors_bar = ["#FFD700" if i == 0 else "#90A4AE" for i in range(len(mape_vals))]
bars = ax2.bar(range(len(mape_vals)), mape_vals.values, color=colors_bar)
ax2.set_xticks(range(len(mape_vals)))
ax2.set_xticklabels(mape_vals.index, rotation=35, ha="right", fontsize=8)
ax2.set_title("MAPE (%) — Plus bas = Meilleur", fontsize=12, fontweight="bold")
for bar, val in zip(bars, mape_vals.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f"{val:.1f}%", ha="center", fontsize=8, fontweight="bold")
ax2.grid(alpha=0.3, axis="y")

# 3. R²
ax3 = fig.add_subplot(gs[1, 1])
r2_vals = df_results["R²"]
bars2 = ax3.bar(range(len(r2_vals)), r2_vals.values, color=colors_bar)
ax3.set_xticks(range(len(r2_vals)))
ax3.set_xticklabels(r2_vals.index, rotation=35, ha="right", fontsize=8)
ax3.set_title("R² — Plus haut = Meilleur", fontsize=12, fontweight="bold")
ax3.axhline(0, color="red", linestyle="--")
for bar, val in zip(bars2, r2_vals.values):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f"{val:.3f}", ha="center", fontsize=8, fontweight="bold")
ax3.grid(alpha=0.3, axis="y")

plt.suptitle("🏆 Comparaison Finale — Revenue Forecasting", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("comparaison_finale.png", dpi=120, bbox_inches="tight")
plt.show()
print("💾 Graphique → comparaison_finale.png")

# Tableau final
df_results.to_csv("comparaison_finale.csv")
print("\n💾 Résultats → comparaison_finale.csv")
print(f"\n🏆 Classement final :")
for i, (model, row) in enumerate(df_results.iterrows()):
    medal = ["🥇", "🥈", "🥉", "  ", "  "][min(i, 4)]
    print(f"   {medal} {model:<25} MAPE: {row['MAPE (%)']:.2f}% | R²: {row['R²']:.4f}")